In [ ]:
# Esame 654AA - a.a. 2025/2026
# Studenti: Leonardo Celati, Samuele Taviano
# Matricole: 660185,

In [ ]:
import importlib
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler

import cup_common as cc
import knn_common as kc
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor


In [ ]:
importlib.reload(cc)
importlib.reload(kc)

In [ ]:
# Split for KFold
n_split = 5
default_cv = KFold(n_splits=n_split, shuffle=True, random_state=42)
default_krange = list(range(1, 41, 2))
default_scoring = "neg_mean_absolute_error"

default_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor())
])

default_param_grid = {
    "knn__n_neighbors": default_krange,
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2],          # 1=Manhattan, 2=Euclidean
    "knn__metric": ["minkowski"]
}




<h2>Cup Dataset</h2>
<hr/>

<h4>Data Loading</h4>
<p>Load and introspect data from monk training and test set.</p>

In [ ]:
df_train, df_test = cc.load_set()

# The train set used for model selection
X_tr, y_tr = cc.prepare_dataset(df_train)
X_ts, y_ts = cc.prepare_dataset(df_test)

#X_tr, y_tr, X_ts, y_ts = cc.prepare_dataset_for_dry_run(df_train, ratio=0.2)
features_names = X_tr.columns
cc.dataset_introspection(df_train, df_test)

In [ ]:
gs = GridSearchCV(
    default_pipe,
    param_grid=default_param_grid,
    scoring=default_scoring,
    cv=default_cv,
    n_jobs=-1,
    return_train_score=True
)

gs.fit(X_tr, y_tr)
model, params = kc.extract_best_knn_metrics_from_grid(gs)

print(params)

In [ ]:
kc.plot_knn_validation_curve_from_gs(gs,"mean")

In [ ]:
#y_tr_np = y_tr.to_numpy()
kc.plot_knn_learning_curve(model, X_tr, y_tr, default_cv, scoring=default_scoring)

In [ ]:
importlib.reload(kc)
kc.plot_knn_learning_curves_grid(
    model,
    X=X_tr,
    y=y_tr,
    cv=default_cv,
    scoring=default_scoring
)

<h4>KFold</h4>

In [ ]:
importlib.reload(kc)
importlib.reload(cc)
best_model = gs.best_estimator_

result = kc.run_kfold(best_model, X_tr, y_tr, default_cv)
print(result)

In [ ]:
kfold_result={"best_vl_mee": result["mee_best"], "vl_mee": result["mee_mean"], "vl_mee_std": result["mee_std"]}
cc.mee_table(kfold_result, "KNN")

In [ ]:
cc.plot_kfold_bar_vl_mee(result["per_fold"]["mee"])

In [ ]:
cc.plot_kfold_bar_vl_rmse(result["per_fold"]["rmse"], use="best")


<h4>Test predictions</h4>
<p>Test predictions on the test set.</p>

In [ ]:
y_pred = model.predict(X_ts)
print(y_pred)